# Sparkle V2 — Fase 3.7: rendimiento medido de cada modo de clase

Los cinco modos de `backend/src/constants/attentionModes.js` fijan el umbral **Y** del alumno,
que es el punto de operación del clasificador. Hasta ahora esos valores estaban elegidos a mano
**sin respaldo medido**. Este notebook mapea cada modo a su rendimiento real.

## Semántica de Y — importa para no equivocar el signo

En `detector.js`: `inattentive = (1 − p) <= Y`, donde `p` es P(desatención) del alumno.
Por lo tanto **la frontera de decisión en el espacio de `p` está en `1 − Y`**, no en 0,5:

| Modo | Y | Frontera en p | Interpretación |
| ---: | ---: | ---: | :--- |
| 1 · Por Defecto | 0,60 | p ≥ 0,40 | marca desatención **antes** |
| 2 · Clase Teórica | 0,50 | p ≥ 0,50 | **el punto en el que se midió la tesis** |
| 3 · Clase Práctica | 0,50 | p ≥ 0,50 | ídem |
| 4 · Seminario / Debate | 0,60 | p ≥ 0,40 | como el 1 |
| 5 · Repaso / Consulta | 0,45 | p ≥ 0,55 | marca desatención **después** |

**Todo lo reportado en Fase 3.6 corresponde a Y = 0,50.** El modo por defecto usa Y = 0,60, así
que un alumno recién instalado opera en un punto que no se midió. Este notebook lo corrige.

**No toca el Test.** Todo se calcula sobre las predicciones out-of-fold de 3.6e.

In [ ]:
import sys, os, json, collections
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2')

OOF_ENS = f'{PROJECT_DIR}/fase3_6_camino70/oof_ensemble.npz'
OUT     = f'{PROJECT_DIR}/fase3_7_modos'
os.makedirs(OUT, exist_ok=True)

import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

K, SPAN_MAX, N_BOOT, SEED = 5, 7, 5000, 42

# Presets tal como están hoy en backend/src/constants/attentionModes.js
MODOS = [
    {'id': 1, 'nombre': 'Por Defecto',        'Y': 0.60, 'Q': 0.80, 'k': 30, 'P': 0.60},
    {'id': 2, 'nombre': 'Clase Teórica',      'Y': 0.50, 'Q': 0.85, 'k': 30, 'P': 0.65},
    {'id': 3, 'nombre': 'Clase Práctica',     'Y': 0.50, 'Q': 0.80, 'k': 25, 'P': 0.60},
    {'id': 4, 'nombre': 'Seminario / Debate', 'Y': 0.60, 'Q': 0.75, 'k': 20, 'P': 0.55},
    {'id': 5, 'nombre': 'Repaso / Consulta',  'Y': 0.45, 'Q': 0.90, 'k': 35, 'P': 0.70},
]

assert os.path.exists(OOF_ENS), f'Falta {OOF_ENS} (lo genera 3.6e)'
z = np.load(OOF_ENS, allow_pickle=True)
p_c = z['p_ens']; y_c = z['y']
part_c = np.array([str(q) for q in z['part']]); ids_c = np.array([str(c) for c in z['ids']])
print(f'OOF: {len(y_c)} clips · {len(set(part_c))} participantes')

## 1. Métricas por modo — nivel clip y nivel ventana

In [ ]:
def parse(cid):
    c = os.path.splitext(str(cid).strip())[0]; return c[:6], int(c[6:])

def ventanas(ids, k=K, span_max=SPAN_MAX):
    por = collections.defaultdict(list)
    for i, cid in enumerate(ids):
        s, t = parse(cid); por[s].append((t, i))
    out = []
    for s, pares in por.items():
        pares.sort(); i = 0
        while i + k - 1 < len(pares):
            v = pares[i:i+k]
            if v[-1][0] - v[0][0] <= span_max - 1:
                out.append([j for _, j in v]); i += k
            else:
                i += 1
    return out

W = ventanas(ids_c)
y_w = np.array([int(y_c[w].sum()*2 > len(w)) for w in W])
p_w = np.array([float(p_c[w].mean()) for w in W])      # media de probabilidades (§9.5)
part_w = np.array([parse(ids_c[w[0]])[0] for w in W])
print(f'{len(W)} ventanas · {len(set(part_w))} participantes')

def metricas(y, p, frontera):
    pred = (p >= frontera).astype(int)
    cm = confusion_matrix(y, pred, normalize='true')
    triv = max((y == 0).mean(), (y == 1).mean())
    acc = accuracy_score(y, pred)
    return {'accuracy': round(float(acc), 4), 'trivial': round(float(triv), 4),
            'brecha': round(float(acc - triv), 4),
            'balanced_acc': round(float(balanced_accuracy_score(y, pred)), 4),
            'f1_macro': round(float(f1_score(y, pred, average='macro')), 4),
            'recall_atento': round(float(cm[0, 0]), 4),
            'recall_desatencion': round(float(cm[1, 1]), 4),
            'falsos_desatento': round(float(cm[0, 1]), 4)}

filas = []
for m in MODOS:
    f = 1 - m['Y']          # frontera en el espacio de p
    filas.append({'modo': m['id'], 'nombre': m['nombre'], 'Y': m['Y'], 'frontera_p': round(f, 2),
                  **{f'clip_{k}': v for k, v in metricas(y_c, p_c, f).items()},
                  **{f'vent_{k}': v for k, v in metricas(y_w, p_w, f).items()}})
df = pd.DataFrame(filas)
pd.set_option('display.width', 200)
print('\n=== POR VENTANA (la unidad del producto) ===')
print(df[['modo', 'nombre', 'Y', 'frontera_p', 'vent_accuracy', 'vent_balanced_acc',
          'vent_recall_atento', 'vent_recall_desatencion', 'vent_falsos_desatento']]
      .to_string(index=False))
print('\n=== POR CLIP ===')
print(df[['modo', 'nombre', 'Y', 'clip_accuracy', 'clip_balanced_acc',
          'clip_recall_atento', 'clip_recall_desatencion']].to_string(index=False))
df.to_csv(f'{OUT}/modos_rendimiento.csv', index=False)

## 2. Curva completa de Y — para elegir con criterio, no a ojo

In [ ]:
Ys = np.round(np.arange(0.30, 0.76, 0.025), 3)
curva = pd.DataFrame([{'Y': Y, **metricas(y_w, p_w, 1 - Y)} for Y in Ys])
print(curva.to_string(index=False))
curva.to_csv(f'{OUT}/curva_Y.csv', index=False)

mejor_ba = curva.loc[curva.balanced_acc.idxmax()]
print(f'\nY que maximiza balanced accuracy: {mejor_ba.Y} (BalAcc {mejor_ba.balanced_acc:.4f})')
print('Nota: "maximiza BalAcc" NO implica "es el mejor para el producto". El costo de un falso '
      '"desatento" (molestar a un alumno atento) y el de una desatención no detectada NO son '
      'simétricos, y esa asimetría es una decisión de producto, no estadística.')

plt.figure(figsize=(7.5, 4.5))
plt.plot(curva.Y, curva.recall_atento, 'o-', label='recall Atento (no molestar al que atiende)')
plt.plot(curva.Y, curva.recall_desatencion, 's-', label='recall Desatención (detectar)')
plt.plot(curva.Y, curva.balanced_acc, '^-', color='k', alpha=.6, label='balanced accuracy')
for m in MODOS:
    plt.axvline(m['Y'], color='gray', ls=':', alpha=.5)
    plt.text(m['Y'], 0.02, f" {m['id']}", fontsize=8, color='gray')
plt.xlabel('Y (umbral de atención del alumno)'); plt.ylim(0, 1)
plt.legend(fontsize=8); plt.title('Compromiso por umbral — medido sobre ventanas de 50 s (OOF)')
plt.tight_layout(); plt.savefig(f'{OUT}/curva_Y.png', dpi=120); plt.show()

## 3. IC clusterizado por participante para los Y de los modos

In [ ]:
def boot(y, p, part, frontera, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); parts = np.unique(part)
    idx = {q: np.where(part == q)[0] for q in parts}
    ok = (p >= frontera).astype(int) == y
    acc, ba = np.empty(n_boot), np.empty(n_boot)
    for i in range(n_boot):
        j = np.concatenate([idx[q] for q in rng.choice(parts, size=len(parts), replace=True)])
        ys, oks = y[j], ok[j]
        n0, n1 = (ys == 0).sum(), (ys == 1).sum()
        acc[i] = oks.mean()
        ba[i] = np.nan if (n0 == 0 or n1 == 0) else 0.5*(oks[ys==0].sum()/n0 + oks[ys==1].sum()/n1)
    pct = lambda v: [round(float(x), 4) for x in np.percentile(v[~np.isnan(v)], [2.5, 97.5])]
    return {'accuracy_IC95': pct(acc), 'balacc_IC95': pct(ba)}

for m in MODOS:
    r = boot(y_w, p_w, part_w, 1 - m['Y'])
    fila = df[df.modo == m['id']].iloc[0]
    print(f"modo {m['id']} · {m['nombre']:20s} Y={m['Y']:.2f} | "
          f"acc {fila.vent_accuracy:.4f} {r['accuracy_IC95']} | "
          f"BalAcc {fila.vent_balanced_acc:.4f} {r['balacc_IC95']}")

## 4. Texto sugerido para el selector de modos

La idea es que el docente elija con una consecuencia a la vista, no con un número abstracto.

In [ ]:
print('=' * 78)
for m in MODOS:
    f = df[df.modo == m['id']].iloc[0]
    fp = f.vent_falsos_desatento * 100
    det = f.vent_recall_desatencion * 100
    print(f"\n【 Modo {m['id']} — {m['nombre']} 】  (Y = {m['Y']:.2f})")
    print(f"  Detecta el {det:.0f} % de la desatención real.")
    print(f"  Marca erróneamente como desatento al {fp:.0f} % de los alumnos que sí atienden.")
    print(f"  Exactitud medida: {f.vent_accuracy*100:.0f} % sobre ventanas de ~1 minuto.")
    if abs(m['Y'] - 0.50) < 1e-9:
        print('  ✔ Es el punto de operación en el que se validó el sistema.')
print('\n' + '=' * 78)

json.dump({'modos': df.to_dict('records'), 'curva_Y': curva.to_dict('records')},
          open(f'{OUT}/modos.json', 'w'), indent=2, ensure_ascii=False, default=str)
print(f'\nGuardado en {OUT}/. Evaluaciones sobre Test: 0.')

## 5. Qué hacer con esto

1. **Revisar el Y del modo "Por Defecto".** Hoy usa 0,60, que no es el punto en el que se
   validó el sistema (0,50). Si la tabla muestra que a 0,60 el recall de *Atento* cae mucho,
   conviene mover el default a 0,50 y dejar 0,60 para los modos que quieran esa agresividad
   de forma deliberada.
2. **Usar los textos de §4 en el selector de modos** en lugar de exponer el número.
3. **Cambiar el agregador de Q.** El producto cuenta decisiones duras ("Q o más de las últimas
   k capturas dieron desatento"). En §9.5 del registro, promediar probabilidades rindió mejor
   que el voto duro (0,6370 vs 0,6282). Conviene promediar `p` sobre la ventana y comparar
   contra la frontera una sola vez.
4. **La abstención debe centrarse en `1 − Y`**, no en 0,50: la banda de confianza tiene que
   seguir a la frontera de decisión del modo activo.